<a href="https://colab.research.google.com/github/uzmaa29/deeplearning/blob/main/customNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

from Scratch: custom neural network with custom optimizers on mnist

this notebook demonstrates how to implement a fully connected neural network with forward and backward propagation from scratch using NumPy, includes custom implementations of key optimization algorithms like stochastic gradient descent (SGD), momentum, RMSprop, Adam, trained and evaluated on the MNIST dataset.

In [1]:
import numpy as np
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

np.random.seed(42)

In [2]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.reshape(-1, 784).astype("float32") / 255.0
X_test = X_test.reshape(-1, 784).astype("float32") / 255.0
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

X_small, y_small = X_train[:20000], y_train[:20000]

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
def relu(x): return np.maximum(0, x)
def relu_d(x): return (x > 0).astype(float)
def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
def sigmoid_d(x): s = sigmoid(x); return s * (1 - s)

def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def cross_entropy(y, p):
    return -np.mean(np.sum(y * np.log(np.clip(p, 1e-10, 1)), axis=1))

In [4]:
class NeuralNetwork:
    def __init__(self, hidden=128, lr=0.001, activation="relu", optimizer="adam"):
        self.lr, self.opt = lr, optimizer
        self.act, self.act_d = (relu, relu_d) if activation == "relu" else (sigmoid, sigmoid_d)
        self.W1 = np.random.randn(784, hidden) * np.sqrt(2 / 784)
        self.b1 = np.zeros((1, hidden))
        self.W2 = np.random.randn(hidden, 10) * np.sqrt(2 / hidden)
        self.b2 = np.zeros((1, 10))
        self.params = ["W1", "b1", "W2", "b2"]
        self.m = {p: np.zeros_like(getattr(self, p)) for p in self.params}  # 1st moment / velocity / cache
        self.v = {p: np.zeros_like(getattr(self, p)) for p in self.params}  # 2nd moment
        self.t = 0

    def forward(self, X):
        self.Z1 = X @ self.W1 + self.b1
        self.A1 = self.act(self.Z1)
        self.A2 = softmax(self.A1 @ self.W2 + self.b2)
        return self.A2

    def backward(self, X, y):
        m = X.shape[0]
        dZ2 = self.A2 - y
        dZ1 = (dZ2 @ self.W2.T) * self.act_d(self.Z1)
        return {
            "W1": X.T @ dZ1 / m, "b1": dZ1.mean(axis=0, keepdims=True),
            "W2": self.A1.T @ dZ2 / m, "b2": dZ2.mean(axis=0, keepdims=True),
        }

    def update(self, grads):
        self.t += 1
        for p in self.params:
            g = grads[p]
            if self.opt == "sgd":
                step = g
            elif self.opt == "momentum":
                self.m[p] = 0.9 * self.m[p] + 0.1 * g
                step = self.m[p]
            elif self.opt == "rmsprop":
                self.v[p] = 0.9 * self.v[p] + 0.1 * g**2
                step = g / (np.sqrt(self.v[p]) + 1e-8)
            else:  # adam
                self.m[p] = 0.9 * self.m[p] + 0.1 * g
                self.v[p] = 0.999 * self.v[p] + 0.001 * g**2
                mh = self.m[p] / (1 - 0.9**self.t)
                vh = self.v[p] / (1 - 0.999**self.t)
                step = mh / (np.sqrt(vh) + 1e-8)
            setattr(self, p, getattr(self, p) - self.lr * step)

    def predict(self, X): return np.argmax(self.forward(X), axis=1)
    def accuracy(self, X, y): return np.mean(self.predict(X) == np.argmax(y, axis=1))

    def train(self, X, y, epochs=5, batch_size=32):
        for epoch in range(epochs):
            idx = np.random.permutation(len(X))
            X, y = X[idx], y[idx]
            losses = []
            for s in range(0, len(X), batch_size):
                Xb, yb = X[s:s+batch_size], y[s:s+batch_size]
                losses.append(cross_entropy(yb, self.forward(Xb)))
                self.update(self.backward(Xb, yb))
            print(f"Epoch {epoch+1}/{epochs} - Loss: {np.mean(losses):.4f} - Accuracy: {self.accuracy(X, y):.4f}")

In [5]:
model = NeuralNetwork()
model.train(X_small, y_small, epochs=10, batch_size=32)
print("Test accuracy:", model.accuracy(X_test, y_test))

Epoch 1/10 - Loss: 0.4080 - Accuracy: 0.9414
Epoch 2/10 - Loss: 0.1846 - Accuracy: 0.9639
Epoch 3/10 - Loss: 0.1272 - Accuracy: 0.9760
Epoch 4/10 - Loss: 0.0930 - Accuracy: 0.9840
Epoch 5/10 - Loss: 0.0722 - Accuracy: 0.9853
Epoch 6/10 - Loss: 0.0532 - Accuracy: 0.9899
Epoch 7/10 - Loss: 0.0419 - Accuracy: 0.9935
Epoch 8/10 - Loss: 0.0323 - Accuracy: 0.9939
Epoch 9/10 - Loss: 0.0254 - Accuracy: 0.9946
Epoch 10/10 - Loss: 0.0191 - Accuracy: 0.9982
Test accuracy: 0.9683


In [6]:
for opt in ["sgd", "momentum", "rmsprop", "adam"]:
    m = NeuralNetwork(lr=0.1 if opt == "sgd" else 0.001, optimizer=opt)
    m.train(X_small, y_small, epochs=5)
    print(opt, m.accuracy(X_test, y_test))

Epoch 1/5 - Loss: 0.4502 - Accuracy: 0.9263
Epoch 2/5 - Loss: 0.2333 - Accuracy: 0.9477
Epoch 3/5 - Loss: 0.1763 - Accuracy: 0.9589
Epoch 4/5 - Loss: 0.1409 - Accuracy: 0.9701
Epoch 5/5 - Loss: 0.1157 - Accuracy: 0.9762
sgd 0.9577
Epoch 1/5 - Loss: 2.1294 - Accuracy: 0.5192
Epoch 2/5 - Loss: 1.6823 - Accuracy: 0.6955
Epoch 3/5 - Loss: 1.3487 - Accuracy: 0.7586
Epoch 4/5 - Loss: 1.1097 - Accuracy: 0.7905
Epoch 5/5 - Loss: 0.9447 - Accuracy: 0.8134
momentum 0.8179
Epoch 1/5 - Loss: 0.3805 - Accuracy: 0.9446
Epoch 2/5 - Loss: 0.1845 - Accuracy: 0.9603
Epoch 3/5 - Loss: 0.1317 - Accuracy: 0.9660
Epoch 4/5 - Loss: 0.1007 - Accuracy: 0.9793
Epoch 5/5 - Loss: 0.0811 - Accuracy: 0.9817
rmsprop 0.9574
Epoch 1/5 - Loss: 0.4106 - Accuracy: 0.9417
Epoch 2/5 - Loss: 0.1889 - Accuracy: 0.9611
Epoch 3/5 - Loss: 0.1309 - Accuracy: 0.9742
Epoch 4/5 - Loss: 0.0935 - Accuracy: 0.9807
Epoch 5/5 - Loss: 0.0721 - Accuracy: 0.9867
adam 0.9626
